
# Submillimeter galaxy SED: dust-obscured starburst at z=3

Dust-obscured starburst at z = 3 with a heavily attenuated diffuse ISM
(`tau_diff = 3.0`). The negative K-correction is what makes submillimeter
selection nearly distance-independent over z ≈ 1–6: as a source recedes, the
observing band walks up the steep Rayleigh-Jeans side of the dust peak, and the
two effects very nearly cancel (Blain+2002).

## References

Blain et al. 2002, PhR, 369, 111 (SMG demographics and K-corrections).


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# Redshift and dust parameters of a typical ALMA/SCUBA-2 SMG
Z_SMG = 3.0
TAU_V = 3.5  # Broad visual absorption → powerful dust-to-star conversion

# Physical parameters: M* = 2e11 Msun, SFR = 500 Msun/yr from SFH
# Implied log_total_mass ≈ 2.7 (peak ~500 Msun/yr)
ssp = tengri.load_ssp()
model = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "dpl",
        "all_params": tengri.FIXED,
        "alpha": 1.5,  # Shallow decay → high SFR at young ages
        "beta": 2.0,  # Steep early-time turnover
        "tau_gyr": 0.8,  # Recent starburst epoch
        "log_total_mass": 10.0,  # Peak SFR = 10^2.7 ≈ 500 Msun/yr
    },
    dust={
        "type": "two_component",
        "all_params": tengri.FIXED,
        "tau_bc": 2.0,  # Birth cloud (young stars)
        "tau_diff": 3.0,  # Diffuse ISM (extended attenuation)
        "emission": {
            "type": "dale2014",
            "all_params": tengri.FIXED,
            "qpah": 3.5,  # Neutral PAH fraction → warmer dust
        },
    },
    redshift=tengri.Fixed(Z_SMG),
)

# Sample parameters from the prior
p = dict(model.spec.sample(jax.random.PRNGKey(0)))

# Predict rest-frame and observed-frame spectra
out_rest = model.predict(p)
wave_rest = np.asarray(model.wavelengths)
sed_rest = np.asarray(out_rest.rest_sed())

# Observed frame: frequency conversion for plotting in nuLnu (SED frame)
wave_obs = wave_rest * (1.0 + Z_SMG)
C_AA_PER_S = 2.998e18
nu_obs = C_AA_PER_S / wave_obs
nu_f_nu = nu_obs * sed_rest

# Create two-panel figure: rest-frame (left) and observed-frame (right)
fig, (ax_rest, ax_obs) = plt.subplots(1, 2, figsize=(12, 4.8))

# === LEFT PANEL: Rest-frame SED ===
# Show why SMGs are "dust-dominated": UV/optical vanish, FIR dominates
nu_rest = C_AA_PER_S / wave_rest
nu_l_nu_rest = nu_rest * sed_rest

mask_rest = sed_rest > 0
ax_rest.loglog(
    wave_rest[mask_rest] / 1e4,  # Convert Angstrom → microns
    nu_l_nu_rest[mask_rest],
    color="C0",
    lw=2.0,
    label="Dust-obscured SED",
)

# Annotate the dust peak (rest-frame ~100 μm for this SFR)
peak_idx = np.argmax(nu_l_nu_rest[mask_rest])
peak_wave = wave_rest[mask_rest][peak_idx]
peak_lnu = nu_l_nu_rest[mask_rest][peak_idx]
ax_rest.annotate(
    f"Dust peak\n~{peak_wave / 1e4:.0f} μm",
    xy=(peak_wave / 1e4, peak_lnu),
    xytext=(peak_wave / 1e4 * 0.1, peak_lnu * 10),
    fontsize=9,
    color="C0",
    arrowprops=dict(arrowstyle="->", color="C0", lw=0.8),
)

# Mark the UV where UV is absorbed
uv_mask = (wave_rest < 3000) & (wave_rest > 1000)
if np.any(uv_mask):
    ax_rest.fill_between(
        wave_rest[uv_mask] / 1e4,
        1e22,
        1e35,
        alpha=0.15,
        color="red",
        label="UV (absorbed)",
    )

ax_rest.set(
    xlim=(0.05, 1e3),
    ylim=(1e22, 1e35),
    xlabel=r"Rest-frame wavelength $\lambda$ [$\mu$m]",
    ylabel=r"$\nu L_\nu$ [erg s$^{-1}$]",
    title=f"Rest-frame (z={Z_SMG}, dust-dominated)",
)
ax_rest.legend(frameon=False, fontsize=9, loc="upper right")

# === RIGHT PANEL: Observed-frame SED ===
# Show the negative K-correction: 850 μm → 3.4 mm keeps SMGs bright
mask_obs = sed_rest > 0
ax_obs.loglog(
    wave_obs[mask_obs] / 1e4,  # Convert Angstrom → microns
    nu_f_nu[mask_obs],
    color="C1",
    lw=2.0,
    label="Observed frame",
)

# Mark the critical 850 μm rest → 3.4 mm observed
wave_850_rest = 850.0 * 1e4  # Convert microns → Angstrom
wave_850_obs = wave_850_rest * (1.0 + Z_SMG)
if np.any(mask_obs) and wave_850_obs > wave_obs[mask_obs].min():
    idx_850 = np.argmin(np.abs(wave_obs[mask_obs] - wave_850_obs))
    ax_obs.plot(
        wave_obs[mask_obs][idx_850] / 1e4,
        nu_f_nu[mask_obs][idx_850],
        "o",
        markersize=10,
        color="orange",
        label="850 μm rest → 3.4 mm obs",
    )
    ax_obs.annotate(
        f"Negative K-correction\nsweetspot: {wave_850_obs / 1e4:.1f} μm",
        xy=(wave_850_obs / 1e4, nu_f_nu[mask_obs][idx_850]),
        xytext=(wave_850_obs / 1e4 * 3, nu_f_nu[mask_obs][idx_850] * 0.3),
        fontsize=9,
        color="orange",
        arrowprops=dict(arrowstyle="->", color="orange", lw=0.8),
    )

# Annotate common (sub)mm band observatories
BANDS_MM = {
    "1.3 mm\nALMA": 1.3e4,
    "0.85 mm\nSCUBA-2": 0.85e4,
}
for name, lam_um in BANDS_MM.items():
    lam_aa = lam_um * 1e4
    if lam_aa > wave_obs[mask_obs].min():
        idx = np.argmin(np.abs(wave_obs[mask_obs] - lam_aa))
        ax_obs.axvline(lam_aa / 1e4, color="gray", lw=0.5, ls="--", alpha=0.5)
        ax_obs.text(
            lam_aa / 1e4,
            1e25,
            name,
            fontsize=8,
            color="0.5",
            ha="center",
            rotation=90,
            va="bottom",
        )

ax_obs.set(
    xlim=(0.2, 1e4),
    ylim=(1e22, 1e35),
    xlabel=r"Observed wavelength $\lambda$ [$\mu$m]",
    ylabel=r"$\nu L_\nu$ [erg s$^{-1}$]",
    title=f"Observed frame (z={Z_SMG}, K-correction visible)",
)
ax_obs.legend(frameon=False, fontsize=9, loc="upper right")

fig.tight_layout()
plt.savefig("plot_submillimeter_galaxy_sed.png", dpi=150, bbox_inches="tight")